# CS 516 Project: EO Fairness Pipeline

**Research question**: Does Equalized Odds (EO) post-processing increase individual-level inconsistency for defendants with similar legal profiles?

**Hypotheses**:
- **H1**: EO post-processing increases individual-level inconsistency relative to the baseline classifier.
- **H2**: EO-induced inconsistencies concentrate among legally similar individuals near decision thresholds.
- **H3**: Reductions in group-level error disparities under EO are negatively associated with individual-level consistency.

**Race encoding decision**: Race is encoded as a binary variable (African-American=1, Caucasian=0), following ProPublica and prior fairness literature. Race is **never** used as a model feature — only for post-processing group assignment and reporting. This analysis is restricted to the two largest racial groups in the dataset to enable direct EO parity measurement.

In [ ]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

# Add src/ to path so modules can be imported regardless of working directory
notebook_dir = os.path.dirname(os.path.abspath('run_pipeline.ipynb'))
project_root = os.path.dirname(notebook_dir)
src_path = os.path.join(project_root, 'src')
for p in [src_path, os.path.join(os.getcwd(), 'src'), 'src']:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.calibration import calibration_curve

from config import (
    RANDOM_SEED, RESULTS_DIR, FIGURES_DIR, BOOTSTRAP_N,
    EPSILON_GRID, K_NEIGHBORS,
    COLOR_BLACK, COLOR_WHITE, COLOR_NEUTRAL,
)

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
})

print('Environment ready. Results will be saved to:', RESULTS_DIR)

## Step 1: Data Preparation

In [ ]:
import data_prep as dp

df = dp.load_and_clean()
print(f'Cleaned dataset: {len(df)} individuals')
print('\nRace distribution:')
print(df['race_raw'].value_counts())
print('\nOutcome distribution:')
print(df['two_year_recid'].value_counts(normalize=True).rename('fraction').to_frame())

In [ ]:
X, X_nr, y = dp.encode_features(df)
race = df['race']
data_split = dp.split_data(X_nr, y, race)

X_train    = data_split['X_train']
X_test     = data_split['X_test']
y_train    = data_split['y_train']
y_test     = data_split['y_test']
race_train = data_split['race_train']
race_test  = data_split['race_test']
idx_test   = data_split['idx_test']

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Test Black: {race_test.sum()} | Test White: {(race_test==0).sum()}')

## Step 2: Baseline Model

In [ ]:
import baseline_model as bm

model = bm.train_baseline(X_train, y_train)
scores_df = bm.score_and_decide(model, X_test)
baseline_metrics = bm.compute_baseline_metrics(y_test, scores_df, race_test)

scores            = scores_df['score']
decisions_baseline = scores_df['decision_baseline']

In [ ]:
# Reliability diagram (calibration sanity check)
fig, ax = plt.subplots(figsize=(5, 5))
prob_true, prob_pred = calibration_curve(y_test, scores, n_bins=10)
ax.plot(prob_pred, prob_true, 'o-', color=COLOR_NEUTRAL, label='Calibrated LR')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Reliability Diagram (Calibration Check)')
ax.legend()
fig.savefig(FIGURES_DIR + 'calibration_check.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 3: EO Post-Processing

In [ ]:
import eo_postprocessor as eop

postprocessor = eop.fit_eo_postprocessor(scores, y_test, race_test, random_seed=RANDOM_SEED)
print('EO postprocessor parameters:')
print(json.dumps(postprocessor, indent=2))

In [ ]:
decisions_eo     = eop.apply_eo_decisions(scores, race_test, postprocessor, random_seed=RANDOM_SEED)
eo_decisions_all = eop.run_multi_seed_eo(scores, race_test, postprocessor, n_seeds=10)
eo_metrics       = eop.compute_eo_metrics(y_test, decisions_eo, race_test)

## Step 4: Similarity Pairs

In [ ]:
import similarity as sim

knn_pairs_eucl = sim.build_knn_pairs(X_test, idx_test, race_test, k=K_NEIGHBORS, metric='euclidean')
print(f'KNN pairs (Euclidean): {len(knn_pairs_eucl)}')
print(knn_pairs_eucl['pair_type'].value_counts())

In [ ]:
knn_pairs_cos = sim.build_knn_pairs(X_test, idx_test, race_test, k=K_NEIGHBORS, metric='cosine')
print(f'KNN pairs (cosine): {len(knn_pairs_cos)}')

In [ ]:
df_test_slice = df.loc[idx_test].copy()

rule_pairs = sim.build_rule_pairs(df_test_slice, race_test)
print(f'Rule-based pairs: {len(rule_pairs)}')
print(rule_pairs['pair_type'].value_counts())

## Step 5: Inconsistency Metrics

In [ ]:
import metrics as met

ir_base = met.inconsistency_with_ci(knn_pairs_eucl, decisions_baseline, n_bootstrap=BOOTSTRAP_N)
ir_eo   = met.inconsistency_with_ci(knn_pairs_eucl, decisions_eo,       n_bootstrap=BOOTSTRAP_N)

print(f"Baseline inconsistency: {ir_base['rate']:.3f} [{ir_base['ci_lower']:.3f}, {ir_base['ci_upper']:.3f}]")
print(f"EO inconsistency:       {ir_eo['rate']:.3f} [{ir_eo['ci_lower']:.3f}, {ir_eo['ci_upper']:.3f}]")
print(f"Delta: {ir_eo['rate'] - ir_base['rate']:+.3f}")

In [ ]:
pair_decomp = met.decompose_inconsistency(
    knn_pairs_eucl, decisions_baseline, decisions_eo, postprocessor, scores
)
print(f"EO-introduced: {pair_decomp['eo_introduced'].sum()} pairs")
print(f"EO-resolved:   {pair_decomp['eo_resolved'].sum()} pairs")

In [ ]:
quartile_df = met.inconsistency_by_threshold_distance(pair_decomp)
print(quartile_df.to_string())

## Step 6: Hypothesis Tests

In [ ]:
import analysis as an

h1_result = an.test_h1(pair_decomp, n_bootstrap=BOOTSTRAP_N)
print(f"H1 delta: {h1_result['delta']:+.4f}")
print(f"95% CI: [{h1_result['ci'][0]:.4f}, {h1_result['ci'][1]:.4f}]")
print(f"H1 supported: {h1_result['h1_supported']}")

In [ ]:
h2_result = an.test_h2(pair_decomp)
print(f"Monotone decreasing: {h2_result['monotone_decreasing']}")
print(f"Spearman r: {h2_result['spearman_r']:.4f}  p: {h2_result['spearman_p']:.4f}")
print(f"H2 supported: {h2_result['h2_supported']}")

In [ ]:
h3_pareto = an.test_h3(
    scores, y_test, race_test, knn_pairs_eucl,
    epsilon_grid=EPSILON_GRID, n_bootstrap=BOOTSTRAP_N,
)
print(h3_pareto.to_string())

## Step 7: Figures

In [ ]:
# Fig 1: ROC curves
fig, ax = plt.subplots(figsize=(6, 5))

fpr_all, tpr_all, _ = roc_curve(y_test, scores)
auc_all = roc_auc_score(y_test, scores)
ax.plot(fpr_all, tpr_all, color=COLOR_NEUTRAL, lw=2, label=f'Overall (AUC={auc_all:.3f})')

mask_b = race_test == 1
fpr_b, tpr_b, _ = roc_curve(y_test[mask_b], scores[mask_b])
auc_b = roc_auc_score(y_test[mask_b], scores[mask_b])
ax.plot(fpr_b, tpr_b, color=COLOR_BLACK, lw=2, label=f'Black (AUC={auc_b:.3f})')

mask_w = race_test == 0
fpr_w, tpr_w, _ = roc_curve(y_test[mask_w], scores[mask_w])
auc_w = roc_auc_score(y_test[mask_w], scores[mask_w])
ax.plot(fpr_w, tpr_w, color=COLOR_WHITE, lw=2, label=f'White (AUC={auc_w:.3f})')

ax.scatter(
    [postprocessor['target_fpr']], [postprocessor['target_tpr']],
    marker='*', s=200, color='red', zorder=5, label='EO target',
)
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Fig 1: ROC Curves — Baseline Logistic Regression')
ax.legend()
fig.savefig(FIGURES_DIR + 'fig1_roc_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Fig 2: Score distribution by race with EO thresholds
fig, ax = plt.subplots(figsize=(7, 4))

scores[race_test == 1].plot.kde(ax=ax, color=COLOR_BLACK, lw=2, label='Black')
scores[race_test == 0].plot.kde(ax=ax, color=COLOR_WHITE, lw=2, label='White')

for group_key, color, lbl in [('group_1', COLOR_BLACK, 'Black EO'), ('group_0', COLOR_WHITE, 'White EO')]:
    p = postprocessor[group_key]
    ax.axvline(p['threshold_low'],  color=color, lw=1.5, ls='--', label=f"{lbl} low={p['threshold_low']:.2f}")
    if abs(p['threshold_high'] - p['threshold_low']) > 0.01:
        ax.axvline(p['threshold_high'], color=color, lw=1.5, ls=':', label=f"{lbl} high={p['threshold_high']:.2f}")

ax.set_xlabel('Calibrated Recidivism Score')
ax.set_ylabel('Density')
ax.set_title('Fig 2: Score Distribution by Race with EO Thresholds')
ax.legend(fontsize=8)
fig.savefig(FIGURES_DIR + 'fig2_score_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Fig 3: Bar chart FPR and FNR — baseline vs EO by race
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

groups  = ['Baseline Black', 'Baseline White', 'EO Black', 'EO White']
colors  = [COLOR_BLACK, COLOR_WHITE, COLOR_BLACK, COLOR_WHITE]
hatches = ['', '', '///', '///']
fpr_vals = [
    baseline_metrics['black']['fpr'], baseline_metrics['white']['fpr'],
    eo_metrics['black']['fpr'],       eo_metrics['white']['fpr'],
]
fnr_vals = [
    baseline_metrics['black']['fnr'], baseline_metrics['white']['fnr'],
    eo_metrics['black']['fnr'],       eo_metrics['white']['fnr'],
]

for ax, metric_name, vals in zip(axes, ['FPR', 'FNR'], [fpr_vals, fnr_vals]):
    bars = ax.bar(groups, vals, color=colors, edgecolor='black', linewidth=0.8)
    for bar, hatch in zip(bars, hatches):
        bar.set_hatch(hatch)
    ax.set_ylabel(metric_name)
    ax.set_title(f'{metric_name}: Baseline vs EO')
    ax.set_xticklabels(groups, rotation=20, ha='right', fontsize=9)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

fig.suptitle('Fig 3: Group Metrics — Baseline vs EO Post-Processing')
fig.tight_layout()
fig.savefig(FIGURES_DIR + 'fig3_group_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Fig 4: Inconsistency by threshold-distance quartile
fig, ax = plt.subplots(figsize=(7, 4))

qdf = quartile_df
x = list(range(len(qdf)))
w = 0.35

ax.bar([i - w/2 for i in x], qdf['inconsistency_rate_baseline'],
       width=w, color=COLOR_NEUTRAL, label='Baseline', edgecolor='black')
ax.bar([i + w/2 for i in x], qdf['inconsistency_rate_eo'],
       width=w, color=COLOR_BLACK, label='EO', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(qdf['quartile'].tolist())
ax.set_xlabel('Threshold-Distance Quartile (Q1 = closest to threshold)')
ax.set_ylabel('Inconsistency Rate')
ax.set_title('Fig 4: Inconsistency Rate by Threshold-Distance Quartile')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
ax.legend()
fig.savefig(FIGURES_DIR + 'fig4_inconsistency_by_quartile.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Fig 5: H3 Pareto frontier (dual y-axis)
fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(h3_pareto['epsilon'], h3_pareto['inconsistency_rate'], 'o-',
         color=COLOR_BLACK, lw=2, label='Inconsistency Rate')
ax1.fill_between(
    h3_pareto['epsilon'], h3_pareto['ci_lower'], h3_pareto['ci_upper'],
    alpha=0.2, color=COLOR_BLACK,
)
ax1.set_xlabel('Epsilon (EO relaxation tolerance)')
ax1.set_ylabel('Inconsistency Rate', color=COLOR_BLACK)
ax1.tick_params(axis='y', labelcolor=COLOR_BLACK)
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

ax2 = ax1.twinx()
ax2.plot(h3_pareto['epsilon'], h3_pareto['fpr_disparity'], 's--',
         color=COLOR_WHITE, lw=2, label='FPR Disparity')
ax2.set_ylabel('FPR Disparity |Black - White|', color=COLOR_WHITE)
ax2.tick_params(axis='y', labelcolor=COLOR_WHITE)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
ax1.set_title('Fig 5: H3 Pareto Frontier — Fairness vs Individual Consistency')
fig.savefig(FIGURES_DIR + 'fig5_pareto_frontier.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Fig 6: Heatmap of EO-introduced inconsistency by (age_bin, prior_bin)
df_ts = df_test_slice.copy()
df_ts['age_bin']   = pd.cut(df_ts['age'],          bins=[0,25,35,100], labels=['18-25','26-35','36+'],  right=True)
df_ts['prior_bin'] = pd.cut(df_ts['priors_count'],  bins=[-1,0,2,5,100], labels=['0','1-2','3-5','6+'], right=True)

pair_prof = pair_decomp.merge(
    df_ts[['age_bin', 'prior_bin']].rename(columns={'age_bin': 'age_bin_i', 'prior_bin': 'prior_bin_i'}),
    left_on='id_i', right_index=True, how='left',
)

heatmap_data = (
    pair_prof.groupby(['age_bin_i', 'prior_bin_i'], observed=True)['eo_introduced']
    .mean().unstack('prior_bin_i')
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    heatmap_data, annot=True, fmt='.2f', cmap='YlOrRd',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'EO-Introduced Inconsistency Rate'},
)
ax.set_xlabel('Prior Offense Bin')
ax.set_ylabel('Age Bin')
ax.set_title('Fig 6: EO-Introduced Inconsistency by Age x Prior Offense Profile')
fig.savefig(FIGURES_DIR + 'fig6_pair_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Fig 7: Seed variance boxplot
seed_rates = []
for col in eo_decisions_all.columns:
    seed_dec = pd.Series(eo_decisions_all[col].values, index=scores.index)
    d_i = seed_dec.loc[knn_pairs_eucl['id_i']].values
    d_j = seed_dec.loc[knn_pairs_eucl['id_j']].values
    seed_rates.append(float((d_i != d_j).mean()))

fig, ax = plt.subplots(figsize=(5, 4))
ax.boxplot(seed_rates, vert=True, patch_artist=True,
           boxprops=dict(facecolor=COLOR_BLACK, alpha=0.6))
ax.axhline(ir_base['rate'], color=COLOR_NEUTRAL, lw=2, ls='--', label='Baseline rate')
ax.set_xlabel('EO decisions (10 random seeds)')
ax.set_ylabel('Inconsistency Rate')
ax.set_title('Fig 7: EO Inconsistency Rate Variance Across 10 Seeds')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
ax.legend()
fig.savefig(FIGURES_DIR + 'fig7_seed_variance.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Seed rates — mean: {np.mean(seed_rates):.3f}, std: {np.std(seed_rates):.4f}')

## Step 8: Tables

In [ ]:
# Table 1: Baseline metrics
t1 = pd.DataFrame([
    {'group': 'Black',   'accuracy': baseline_metrics['black']['accuracy'],
     'auc_roc': baseline_metrics['overall']['auc_roc'],
     'fpr': baseline_metrics['black']['fpr'], 'fnr': baseline_metrics['black']['fnr']},
    {'group': 'White',   'accuracy': baseline_metrics['white']['accuracy'],
     'auc_roc': baseline_metrics['overall']['auc_roc'],
     'fpr': baseline_metrics['white']['fpr'], 'fnr': baseline_metrics['white']['fnr']},
    {'group': 'Overall', 'accuracy': baseline_metrics['overall']['accuracy'],
     'auc_roc': baseline_metrics['overall']['auc_roc'], 'fpr': None, 'fnr': None},
])
t1.to_csv(RESULTS_DIR + 'table1_baseline_metrics.csv', index=False)
print('Table 1: Baseline Metrics')
display(t1)

In [ ]:
# Table 2: EO metrics
t2 = pd.DataFrame([
    {'group': 'Black',   'accuracy': eo_metrics['black']['accuracy'],
     'fpr': eo_metrics['black']['fpr'], 'fnr': eo_metrics['black']['fnr']},
    {'group': 'White',   'accuracy': eo_metrics['white']['accuracy'],
     'fpr': eo_metrics['white']['fpr'], 'fnr': eo_metrics['white']['fnr']},
    {'group': 'Overall', 'accuracy': eo_metrics['overall']['accuracy'],
     'fpr': None, 'fnr': None},
])
t2.to_csv(RESULTS_DIR + 'table2_eo_metrics.csv', index=False)
print('Table 2: EO Metrics')
display(t2)

In [ ]:
# Table 3: Hypothesis results summary
t3 = pd.DataFrame([
    {'hypothesis': 'H1',
     'description': 'EO increases inconsistency',
     'stat': f"{h1_result['delta']:+.4f}",
     'ci_or_p': f"[{h1_result['ci'][0]:.4f}, {h1_result['ci'][1]:.4f}]",
     'supported': h1_result['h1_supported']},
    {'hypothesis': 'H2',
     'description': 'Inconsistency concentrates near threshold',
     'stat': f"Spearman r={h2_result['spearman_r']:.4f}",
     'ci_or_p': f"p={h2_result['spearman_p']:.4f}",
     'supported': h2_result['h2_supported']},
    {'hypothesis': 'H3',
     'description': 'Fairness-consistency tradeoff',
     'stat': 'See h3_pareto.csv / Fig 5',
     'ci_or_p': '',
     'supported': 'Visual'},
])
t3.to_csv(RESULTS_DIR + 'table3_hypothesis_results.csv', index=False)
print('Table 3: Hypothesis Results')
display(t3)

In [ ]:
# Table 4: Robustness checks
print('Running robustness checks (may take several minutes)...')
robustness_results = an.robustness_checks(data_split, scores, epsilon_grid=EPSILON_GRID)

t4_rows = [{'variant': 'Primary (LR + Euclidean KNN)', 'h1_delta': h1_result['delta']}]
for variant, res in robustness_results.items():
    t4_rows.append({'variant': variant, 'h1_delta': res['h1_delta']})

t4 = pd.DataFrame(t4_rows)
t4.to_csv(RESULTS_DIR + 'table4_robustness.csv', index=False)
print('Table 4: Robustness Checks')
display(t4)

## Summary

All results saved to `results/`.

| File | Content |
|---|---|
| `results/figures/fig1_roc_comparison.png` | ROC curves by group |
| `results/figures/fig2_score_distribution.png` | Score densities with EO thresholds |
| `results/figures/fig3_group_metrics.png` | FPR/FNR bar chart |
| `results/figures/fig4_inconsistency_by_quartile.png` | Threshold-distance quartile analysis |
| `results/figures/fig5_pareto_frontier.png` | H3 Pareto curve |
| `results/figures/fig6_pair_heatmap.png` | EO inconsistency by profile |
| `results/figures/fig7_seed_variance.png` | Seed variance boxplot |
| `results/table1_baseline_metrics.csv` | Baseline model metrics |
| `results/table2_eo_metrics.csv` | EO model metrics |
| `results/table3_hypothesis_results.csv` | H1–H3 test results |
| `results/table4_robustness.csv` | Robustness variants |
| `results/h3_pareto.csv` | Full H3 sweep data |